# Kev 800M: best.pt of every paper run on Grummon 2025, Kreps 2020 (COVID), Kantorowicz 2026, and Magistro 2025

What and why: `README.md` in this folder.

In [1]:
# Environment flags, set before any Hugging Face or torch import (same as slurm/train-notebook.sbatch)
import os
from pathlib import Path

# On a cluster (Slurm job, incl. JupyterLab sessions), use the shared Hugging Face cache on the project space
ON_CLUSTER = "SLURM_JOB_ID" in os.environ
CLUSTER_HF_HOME = Path.home() / "links" / "projects" / "def-yuma" / "wadi" / "hf"
if ON_CLUSTER:
    if not CLUSTER_HF_HOME.is_dir():
        raise FileNotFoundError(f"Cluster Hugging Face cache not found: {CLUSTER_HF_HOME}")
    os.environ["HF_HOME"] = str(CLUSTER_HF_HOME)

# Never download: the base model must already be in the Hugging Face cache
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Lets the CUDA allocator grow segments instead of fragmenting
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"ON_CLUSTER: {ON_CLUSTER}")
if ON_CLUSTER:
    print(f"HF_HOME: {os.environ['HF_HOME']}")

ON_CLUSTER: False


In [2]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
PAPER_DIR = BETA_TRAINING_DIR / "ms_paper_beta_loss"

# Run from `code/` so data loaders and shared disk caches resolve identically to the training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR), str(PAPER_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_train import runs as run_store
from beta_train.model_state import reset_trainable, set_checkpointing, snapshot_trainable
from beta_train.progress import PROGRESS_INTERVAL_SEC, HtmlProgress
from beta_train.validation import PopulationValidator, validate_all, validate_all_cached, validation_metrics_frame

NOTEBOOK = "04-test-best-800m"
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"
MODEL_SLUG = "kev-0.8b"
# Runs pulled from Rorqual by pull-rorqual-runs.sh (run files + checkpoints/best.pt)
RUNS_DIR = PAPER_DIR / "runs-rorqual" / MODEL_SLUG
# Each run's test results go in <run folder>/test_best/
TEST_FOLDER_NAME = "test_best"
ROW1_DIR = RUNS_DIR / "row1_base_untrained"

print(f"RUNS_DIR: {RUNS_DIR}")


RUNS_DIR: /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/runs-rorqual/kev-0.8b


In [3]:
# Same backbone and LoRA shape as the training notebooks, so best.pt loads into it
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
device = "cuda"
free_gpu()

KEV_BASE = "jaredpalmer/kev-0.8b@bf75a6a8848ea6960ff2ed108d9ed44c2941174f"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0

kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
set_checkpointing(kev_model, False)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
# Same key as the training notebooks, so their cached step-0 validations are reused
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"{KEV_BASE}: {len(INIT_TRAINABLE_STATE)} trainable tensors")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

jaredpalmer/kev-0.8b@bf75a6a8848ea6960ff2ed108d9ed44c2941174f: 376 trainable tensors


## Test datasets

In [6]:
TEST_DATASETS = ["grummon2025", "covid", "kantorowicz2026", "magistro2025"]
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {}
for dataset_name in TEST_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = dataset_module.load_choice_df()
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    validators[dataset_name] = PopulationValidator(
        dataset_module,
        choice_df,
        specs,
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))


# Writes metrics.csv (one row per test dataset), betas.csv (model vs human betas, stacked) and test.json (what was tested)
def save_test_results(test_dir: Path, validation: dict, test_info: dict) -> None:
    test_dir.mkdir(parents=True, exist_ok=True)
    metrics_df = validation_metrics_frame(validation).T
    metrics_df.index.name = "dataset"
    run_store.write_text_atomic(test_dir / "metrics.csv", metrics_df.to_csv())
    run_store.write_betas_file(test_dir / "betas.csv", validation)
    run_store.write_text_atomic(test_dir / "test.json", json.dumps(test_info, indent=2))


# Saves every dataset in `results`: the TEST_DATASETS finished so far (in TEST_DATASETS order), then any earlier-saved
# dataset no longer in TEST_DATASETS (kept, never deleted). test.json lists exactly these, so a rerun reuses them and
# evaluates the rest. Returns the saved results.
def save_merged_test_results(test_dir: Path, results: dict, test_info: dict) -> dict:
    current = [d for d in TEST_DATASETS if d in results]
    kept = [d for d in results if d not in TEST_DATASETS]
    merged = {d: results[d] for d in current + kept}
    merged_info = dict(test_info)
    merged_info["test_datasets"] = list(merged)
    save_test_results(test_dir, merged, merged_info)
    return merged


# True when test_dir already holds results for this checkpoint step and model config on every dataset in TEST_DATASETS
# (other saved datasets do not matter)
def already_tested(test_dir: Path, checkpoint_step: int) -> bool:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return False
    stored = json.loads(test_json.read_text())
    if stored["checkpoint_step"] != checkpoint_step or stored["model"] != MODEL_KEY:
        return False
    missing = [d for d in TEST_DATASETS if d not in stored["test_datasets"]]
    return len(missing) == 0


# Loads every already-saved dataset result when test_dir matches this checkpoint step and model config, so adding a
# new dataset to TEST_DATASETS evaluates only the missing dataset(s) and merges cleanly (datasets no longer in
# TEST_DATASETS are loaded too, so they are kept on the next save).
# test.json is written last, so it is the record of what is complete: metrics.csv and betas.csv must hold every
# dataset it lists; an extra dataset in them (a save stopped before test.json was written) is ignored.
def load_existing_test_results(
    test_dir: Path, checkpoint_step: int
) -> dict[str, tuple[dict[str, float], pd.DataFrame]]:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return {}
    metrics_csv = test_dir / "metrics.csv"
    betas_csv = test_dir / "betas.csv"
    if not metrics_csv.exists() or not betas_csv.exists():
        raise FileNotFoundError(f"Incomplete test_best directory in {test_dir}")
    stored = json.loads(test_json.read_text())
    if stored["checkpoint_step"] != checkpoint_step or stored["model"] != MODEL_KEY:
        return {}
    stored_datasets = list(stored["test_datasets"])
    metrics_df = pd.read_csv(metrics_csv, float_precision="round_trip").set_index("dataset")
    beta_dfs = run_store.read_betas_file(betas_csv)
    not_in_metrics = [ds for ds in stored_datasets if ds not in metrics_df.index]
    not_in_betas = [ds for ds in stored_datasets if ds not in beta_dfs]
    if not_in_metrics or not_in_betas:
        raise ValueError(
            f"Inconsistent test_best artifacts in {test_dir}: test.json lists {stored_datasets}, "
            f"but metrics.csv lacks {not_in_metrics} and betas.csv lacks {not_in_betas}"
        )
    return {
        ds: ({k: float(v) for k, v in metrics_df.loc[ds].items()}, beta_dfs[ds])
        for ds in stored_datasets
    }


,kind,tasks,packed_persona_states,regressions,max_questions,max_chars,graph_limits,prep_sec
validator,,,,,,,,
grummon2025,fused,33798,4906,grummon2025,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",6.574
covid,fused,9855,1845,covid,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",1.845
kantorowicz2026,fused,26823,42,"kantorowicz2026/computers, kantorowicz2026/lau...",1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",3.230
magistro2025,fused,21005,2221,"magistro2025/us_ai, magistro2025/us_offshoring...",1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",3.915


In [5]:
beta_ols.DATASET_MODULES

{'wu2025': <module 'beta_ols.wu2025' from '/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/beta_ols/wu2025.py'>,
 'berke2023': <module 'beta_ols.berke2023' from '/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/beta_ols/berke2023.py'>,
 'grummon2023_calories': <module 'beta_ols.grummon2023_calories' from '/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/beta_ols/grummon2023_calories.py'>,
 'grummon2023_labels': <module 'beta_ols.grummon2023_labels' from '/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/beta_ols/grummon2023_labels.py'>,
 'kriner2026': <module 'beta_ols.kriner2026' from '/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/beta_ols/kriner2026.py'>,
 'covid': <module 'beta_ols.covid' from '/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/beta_ols/covid.py'>,
 'grummon2025': <module 'beta_ols.grummon2025'

## Row 1: untrained model

In [ ]:
# Untrained model (step 0): reuse existing test_best/ results when step and model match,
# and evaluate only missing datasets (cached in code/.cache/beta_training)
row1_test_dir = ROW1_DIR / TEST_FOLDER_NAME
if already_tested(row1_test_dir, 0):
    step0_validation = load_existing_test_results(row1_test_dir, 0)
    print("Row 1 untrained (step 0): already tested, loaded from test_best/")
else:
    existing_step0 = load_existing_test_results(row1_test_dir, 0)
    missing_datasets = [d for d in TEST_DATASETS if d not in existing_step0]
    step0_info = {
        "notebook": NOTEBOOK,
        "model": MODEL_KEY,
        "row": "Row 1",
        "checkpoint": "untrained",
        "checkpoint_step": 0,
        "test_datasets": TEST_DATASETS,
    }
    # One missing dataset at a time, saved as soon as it finishes (a stopped job keeps the datasets it finished)
    step0_results = dict(existing_step0)
    for dataset_name in missing_datasets:
        new_step0, step0_cache_report = validate_all_cached(
            kev_model,
            {dataset_name: validators[dataset_name]},
            INIT_TRAINABLE_STATE,
            model_key=MODEL_KEY,
            cache_dir=CACHE_DIR,
        )
        display(step0_cache_report)
        step0_results[dataset_name] = new_step0[dataset_name]
        save_merged_test_results(row1_test_dir, step0_results, step0_info)
    step0_validation = save_merged_test_results(row1_test_dir, step0_results, step0_info)
    reused = [d for d in TEST_DATASETS if d not in missing_datasets]
    note = f"tested {missing_datasets}, reused {reused}" if reused else f"tested {missing_datasets}"
    print(f"Row 1 untrained (step 0): {note}")
display(validation_metrics_frame(step0_validation).round(5))


## best.pt of every finished run

In [5]:
# best.pt of every finished run in RUNS_DIR; unfinished runs are skipped (rerun this notebook once they finish)
progress_bar = HtmlProgress(
    total=0, initial=0, description="test (waiting)", unit="batch", min_interval_sec=PROGRESS_INTERVAL_SEC
)
for run_dir in sorted(RUNS_DIR.iterdir()):
    if not (run_dir / "run.json").exists():
        continue
    run_info = run_store.read_run_json(run_dir)
    row_code = run_info["identity"]["run_metadata"]["paper_row"]
    if run_info["status"] != "finished":
        print(f"{row_code} {run_dir.name}: {run_info['status']}, skipped")
        continue
    test_dir = run_dir / TEST_FOLDER_NAME
    checkpoint = run_store.load_best(run_dir)
    best = checkpoint["best"]
    checkpoint_step = int(best["step"])
    if already_tested(test_dir, checkpoint_step):
        print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) already tested, skipped")
        continue
    existing = load_existing_test_results(test_dir, checkpoint_step)
    missing_datasets = [d for d in TEST_DATASETS if d not in existing]
    test_info = {
        "notebook": NOTEBOOK,
        "model": MODEL_KEY,
        "row": row_code,
        "checkpoint": "best.pt",
        "checkpoint_step": checkpoint_step,
        "best_metric": best["metric"],
        "best_value": float(best["value"]),
        "test_datasets": TEST_DATASETS,
    }
    # One missing dataset at a time, saved as soon as it finishes (a stopped job keeps the datasets it finished)
    results = dict(existing)
    if missing_datasets:
        reset_trainable(kev_model, checkpoint["trainable"])
    for dataset_name in missing_datasets:
        new_result = validate_all(kev_model, {dataset_name: validators[dataset_name]}, progress_bar)
        results[dataset_name] = new_result[dataset_name]
        save_merged_test_results(test_dir, results, test_info)
    validation = save_merged_test_results(test_dir, results, test_info)
    reused = [d for d in TEST_DATASETS if d not in missing_datasets]
    note = f"tested {missing_datasets}, reused {reused}" if reused else f"tested {missing_datasets}"
    print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) {note}")
    display(validation_metrics_frame(validation).round(5))
progress_bar.close()
reset_trainable(kev_model, INIT_TRAINABLE_STATE)


Row 2 (Ablation) row2_ablation_wu_table_no_kl: best.pt (step 576) tested ['kantorowicz2026'], reused ['grummon2025', 'covid']


,grummon2025,covid,kantorowicz2026
kl_loss,1.02090,1.17363,1.00426
beta_mse,0.02690,0.02046,0.00899
beta_mae,0.16078,0.09866,0.07354
beta_mape_pct,156.75805,233.67983,168.79968
spearman_rho,0.70000,0.66364,0.05788
eval_prep_sec,0.03094,0.00843,0.41898
eval_gpu_sec,60.61859,27.05923,64.26054
eval_wall_sec,62.38241,28.67500,67.30778
eval_ms_per_task,1.84574,2.90969,2.50933


Row 2 row2_wu_table_anchored: best.pt (step 864) tested ['kantorowicz2026'], reused ['grummon2025', 'covid']


,grummon2025,covid,kantorowicz2026
kl_loss,0.96538,1.14068,0.90453
beta_mse,0.00731,0.01335,0.00843
beta_mae,0.08538,0.08946,0.06939
beta_mape_pct,82.30312,193.66948,136.60328
spearman_rho,0.70000,0.69091,-0.05351
eval_prep_sec,0.02610,0.00879,0.02126
eval_gpu_sec,60.67761,27.08822,64.30050
eval_wall_sec,62.38624,28.85716,66.01920
eval_ms_per_task,1.84586,2.92818,2.46129


Row 2b row2b_wu_ce_oracle: best.pt (step 640) tested ['kantorowicz2026'], reused ['grummon2025', 'covid']


,grummon2025,covid,kantorowicz2026
kl_loss,0.91534,1.08210,0.82102
beta_mse,0.00353,0.00373,0.00735
beta_mae,0.05163,0.05044,0.06261
beta_mape_pct,40.33992,108.52474,103.18946
spearman_rho,0.50000,0.79091,0.37039
eval_prep_sec,0.03021,0.00942,0.02063
eval_gpu_sec,60.74558,27.03404,64.27948
eval_wall_sec,62.51586,28.62889,65.90605
eval_ms_per_task,1.84969,2.90501,2.45707


... (4 outputs left out)

Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl: best.pt (step 2240) tested ['kantorowicz2026'], reused ['grummon2025', 'covid']


,grummon2025,covid,kantorowicz2026
kl_loss,0.87411,1.08414,0.72334
beta_mse,0.00425,0.00463,0.00607
beta_mae,0.04941,0.05803,0.05849
beta_mape_pct,50.45800,129.45590,104.35891
spearman_rho,0.60000,0.63636,0.41722
eval_prep_sec,0.02751,0.01033,0.02220
eval_gpu_sec,60.74998,27.09469,64.34406
eval_wall_sec,62.63950,28.98938,65.94500
eval_ms_per_task,1.85335,2.94159,2.45852


Row 5 row5_full_berke_wu_ce_oracle: best.pt (step 1600) tested ['kantorowicz2026'], reused ['grummon2025', 'covid']


,grummon2025,covid,kantorowicz2026
kl_loss,0.87928,1.06686,0.69459
beta_mse,0.00230,0.00256,0.00604
beta_mae,0.03735,0.03936,0.05691
beta_mape_pct,34.33993,84.99044,94.05790
spearman_rho,0.50000,0.82727,0.49316
eval_prep_sec,0.02868,0.01010,0.01964
eval_gpu_sec,60.76075,27.13302,64.28237
eval_wall_sec,62.87645,29.12337,65.86054
eval_ms_per_task,1.86036,2.95519,2.45538


## Summary

In [ ]:
# Every tested checkpoint of this model: one row per (row, run folder, test dataset)
summary_frames = []
for test_json in sorted(RUNS_DIR.glob(f"*/{TEST_FOLDER_NAME}/test.json")):
    test_dir = test_json.parent
    test_info = json.loads(test_json.read_text())
    metrics_df = pd.read_csv(test_dir / "metrics.csv")
    metrics_df.insert(0, "checkpoint_step", test_info["checkpoint_step"])
    metrics_df.insert(0, "slug", test_dir.parent.name)
    metrics_df.insert(0, "row", test_info["row"])
    summary_frames.append(metrics_df)
summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df = summary_df.sort_values(["dataset", "row", "slug"]).set_index(["dataset", "row", "slug"])
display(summary_df[["checkpoint_step", "beta_mse", "beta_mae", "spearman_rho", "kl_loss"]].round(5))


checkpoint_step  beta_mse  beta_mae  spearman_rho  kl_loss
dataset         row              slug                                                                                              
covid           Row 1            row1_base_untrained                                   0   0.01207   0.08735       0.55455  1.18456
                Row 2            row2_wu_table_anchored                              864   0.01335   0.08946       0.69091  1.14068
                Row 2 (Ablation) row2_ablation_wu_table_no_kl                        576   0.02046   0.09866       0.66364  1.17363
                Row 2b           row2b_wu_ce_oracle                                  640   0.00373   0.05044       0.79091  1.08210
                Row 3            row3_berke_ce_only                                 1152   0.00408   0.05298       0.70909  1.10543
                Row 4            row4_hybrid_berke_ce_wu_table_anchored             2240   0.00442   0.05586       0.63636  1.09213
                Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl                2240   0.00463   0.05803       0.63636  1.08414
                Row 5            row5_full_berke_wu_ce_oracle                       1600   0.00256   0.03936       0.82727  1.06686
grummon2025     Row 1            row1_base_untrained                                   0   0.00546   0.06510       0.50000  1.05890
                Row 2            row2_wu_table_anchored                              864   0.00731   0.08538       0.70000  0.96538
                Row 2 (Ablation) row2_ablation_wu_table_no_kl                        576   0.02690   0.16078       0.70000  1.02090
                Row 2b           row2b_wu_ce_oracle                                  640   0.00353   0.05163       0.50000  0.91534
                Row 3            row3_berke_ce_only                                 1152   0.00325   0.05029       0.50000  0.95350
                Row 4            row4_hybrid_berke_ce_wu_table_anchored             2240   0.00355   0.04600       0.60000  0.91100
                Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl                2240   0.00425   0.04941       0.60000  0.87411
                Row 5            row5_full_berke_wu_ce_oracle                       1600   0.00230   0.03735       0.50000  0.87928
kantorowicz2026 Row 1            row1_base_untrained                                   0   0.00834   0.06718      -0.03119  1.41834
                Row 2            row2_wu_table_anchored                              864   0.00843   0.06939      -0.05351  0.90453
                Row 2 (Ablation) row2_ablation_wu_table_no_kl                        576   0.00899   0.07354       0.05788  1.00426
                Row 2b           row2b_wu_ce_oracle                                  640   0.00735   0.06261       0.37039  0.82102
                Row 3            row3_berke_ce_only                                 1152   0.00717   0.06269       0.14783  0.80567
                Row 4            row4_hybrid_berke_ce_wu_table_anchored             2240   0.00672   0.06045       0.39009  0.75342
                Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl                2240   0.00607   0.05849       0.41722  0.72334
                Row 5            row5_full_berke_wu_ce_oracle                       1600   0.00604   0.05691       0.49316  0.69459
magistro2025    Row 1            row1_base_untrained                                   0   0.03392   0.15094       0.36840  2.28346
                Row 2            row2_wu_table_anchored                              864   0.02981   0.14591       0.55205  1.75905
                Row 2 (Ablation) row2_ablation_wu_table_no_kl                        576   0.03316   0.15287       0.50587  2.06854
                Row 2b           row2b_wu_ce_oracle                                  640   0.02960   0.14571       0.54802  1.68919
                Row 3            row3_berke_ce_only                                 1152   0.03441   0.15386     

: 